# Chapter 3: Sets, Dictionaries, and Hashing

**Colab slideshow notebook**  
Based on Chapter 3 of the book "Computational Thinking for Life Scientists"
by Benny Chor and Amir Rubinstein.

## Part II: Sequences — “String along”

Biological sequences are strings over biological alphabets.

Examples:

- DNA alphabet: `{A, C, G, T}`
- Amino-acid alphabet: 20 characters

In this chapter we focus on efficient storage and retrieval of substrings using hashing.

## Chapter goals

By the end of this chapter, we want to understand:

- why naïve search can be too slow,
- how Python `set` supports efficient lookup,
- what hash functions and hash tables are,
- how dictionaries help count k-mers,
- how memory can become a limiting factor.

## The common substring problem

**Input:** two sequences `s1`, `s2`, and a positive integer `k`.

**Output:** a common substring of `s1` and `s2` of length `k`, if one exists.

In biology, a substring of length `k` is often called a **k-mer**.

## Common k-mer
<img src="fig3.0.png" width="600">

Two DNA strings, with matching common k-mer.

In [1]:
# Code snippet: read the Mycobacterium tuberculosis genome
import os
import urllib.request

os.makedirs("data", exist_ok=True)

# Use GitHub’s special server 
# to access raw file contents directly:
BASE_URL = "https://raw.githubusercontent.com/ariefouren/BIU-bio-python-2026/main/data"

files = [
    "Mycobacterium_tuberculosis.txt",
    "Salmonella_enterica.txt",
    "Mycobacterium_leprae.txt",
]

for filename in files:  # loop through the list of files
    url = f"{BASE_URL}/{filename}"
    local_path = f"data/{filename}"

    if not os.path.exists(local_path):
        # download the file from the URL 
        # and save it to the local path
        urllib.request.urlretrieve(url, local_path) 
        print("Downloaded:", filename)
    else:
        print("Already exists:", filename)

# with open(...) as f:  (context manager)
# 1. Python opens the file.
# 2. The opened file object is stored in the variable f.
# 3. The indented block runs.
# 4. When the block ends, Python automatically closes the file.
with open("data/Mycobacterium_tuberculosis.txt") as f:
    Mycobacterium_tuberculosis = f.read().replace("\n", "")

with open("data/Salmonella_enterica.txt") as f:
    Salmonella_enterica = f.read().replace("\n", "")

with open("data/Mycobacterium_leprae.txt") as f:
    Mycobacterium_leprae = f.read().replace("\n", "")
    
    

Downloaded: Mycobacterium_tuberculosis.txt
Downloaded: Salmonella_enterica.txt
Downloaded: Mycobacterium_leprae.txt


In [2]:
# Code snippet: check genome lengths

n1 = len(Mycobacterium_tuberculosis)
n2 = len(Salmonella_enterica)
n3 = len(Mycobacterium_tuberculosis)
print("Mycobacterium tuberculosis genome length:", n1)
print("Salmonella enterica genome length:", n2)
print("Mycobacterium tuberculosis genome length:", n3)


Mycobacterium tuberculosis genome length: 4411532
Salmonella enterica genome length: 4809037
Mycobacterium tuberculosis genome length: 4411532


In [ ]:
# Code snippet: print only a short slice, not the whole genome

Mycobacterium_tuberculosis[:100] # print the first 100 characters 

## Exercise 1

`AGGGG` is also a common substring to both genomes.

1. Find the first position where `AGGGG` appears in each genome using `find`.
2. Count how many times it appears in each genome using `count`.

In [ ]:
# Code snippet: naïve common substring search

def common_substring_naive(s1, s2, k):
    ''' find a common substring of length k in both s1 and s2 '''
    for i in range(len(s1)-k+1):
        for j in range(len(s2)-k+1):
            if s1[i:i+k] == s2[j:j+k]:
                return s1[i:i+k] # return first match found
    return None

In [ ]:
# Code snippet: running example with k = 10

common_substring_naive(Mycobacterium_tuberculosis, Salmonella_enterica, 10)

## Exercise 2

Modify `common_substring_naive` so it returns:

```python
(matching_kmer, i, j)
```

where `i` is the starting index in `s1` and `j` is the starting index in `s2`.

## Exercise 3

Modify the naïve solution to return **all** matching k-mers, not just the first.

Example:

```python
common_substring_all_naive("AGGAT", "GGACGAT", 3)
# ['GGA', 'GAT']
```

In [ ]:
# Code snippet: do NOT run this on large genomes in class
# It may take a very long time for k = 100.

common_substring_naive(Mycobacterium_tuberculosis, Salmonella_enterica, 100)

In [ ]:
# Code snippet: naïve solution with diagnostic printouts

def common_substring_naive(s1, s2, k):
    ''' find a common substring of s1 and s2 of length k '''
    for i in range(len(s1)-k+1):
        for j in range(len(s2)-k+1):
            if s1[i:i+k] == s2[j:j+k]:
                return s1[i:i+k]
        print(i) # added this line!
    return None

## Key point

The naïve algorithm compares many pairs of k-mers.

Worst case:

```text
(number of k-mers in s1) × (number of k-mers in s2)
```

This can be enormous for bacterial genomes.

If each genome has 5 million k-mers, we have 25 trillion comparisons!
It will take about 51 days on a 1 GHz computer.



## Why is the naïve algorithm so slow?
For `k=10` the naïve algorithm algorithm finds the common 10-mer very fast (for i = 0, j = 5940).

For `k=100` there are no common 100-mers, but the algorithm still needs to check all pairs of 100-mers, which takes a very long time. 

In terms of complexity, it is the worst case !

## Python sets to the Rescue 
A Python `set` is a collection of unique elements that supports efficient membership testing.

To solve the common substring problem efficiently, we can use a set to store all k-mers of one genome and then check for each k-mer of the other genome if it is in the set.

We use the following set operations:
- `add`: to add an element to the set.
- `in`: to check if an element is in the set.

In [ ]:
# Code snippet: hash-based solution using Python sets

def common_substring_hash(s1, s2, k):
    ''' find a common length k substring of s1 and s2
        using Python built-in sets '''
    table = set()       # create an empty set to store all k-mers of s1
    for i in range(len(s1)-k+1): 
        table.add(s1[i:i+k])

    for i in range(len(s2)-k+1):    # scan through s2 and check if any k-mer is in the set
        if s2[i:i+k] in table:
            return s2[i:i+k]

    return None

In [ ]:
# Code snippet: test the hash-based solution
import time
start_time = time.time()
common_k_mer = common_substring_hash(Mycobacterium_tuberculosis, Salmonella_enterica, 30)
end_time = time.time()
print("Common k-mer found:", common_k_mer)
print("Time taken:", end_time - start_time, "seconds")

## 3.2. Hash Functions and Hash Tables
### 3.2.1. Hash Functions

## Hash function

<img src="fig3.1.png" width="600">

**Figure 3.1** a large universe `U` mapped into a hash table `T` with several slots using a hash function `h`.

In [ ]:
# Code snippet: a hash function for strings

def hash4strings(st):
    s = 0
    for c in st:
        s = (128*s + ord(c)) % (2**120+451)
    return s**2 % (2**120+451)

In [ ]:
# Code snippet: Unicode values of characters

ord("a"), ord("A"), ord(" ")

In [ ]:
# Code snippet: hash4strings examples

hash4strings("A"), hash4strings("ATTA"), hash4strings("GTTA")

In [ ]:
# Code snippet: Python's built-in hash function
# Important: results vary between sessions and machines.

hash(3), hash(3**100), hash(5.9), hash("A"), hash("ATTA"), hash("GTTA")

## Collisions

A collision occurs when two different elements are mapped to the same table index.

This is unavoidable when a large universe is mapped into a smaller finite table.

In [ ]:
# Code snippet: collision modulo table size
# Important: exact hash values vary between sessions.

hash("ATBGAAA") % 7 == hash("Benny") % 7

## Collision between two elements
<img src="fig3.2.png" width="600">

**Figure 3.2** collision of two different elements.

## Chaining to resolve collisions 
<img src="fig3.3.png" width="600">

**Figure 3.3 placeholder:** chaining to resolve collisions.

## Open addressing to resolve collisions

<img src="fig3.4.png" width="600">

**Figure 3.4** Using open addressing to resolve collisions.

## Time complexity of the Hash-based solution

The structure of the hash-based solution is as follows:

1. Create an empty hash table of size m
2. Insert all k-mers of s1 into hash table (the address is hash(string) % m)
3. For every k-mer of s2,
    if it is in the hash table – return it and finish
4. Declare no common substring of length k

Naïve solution, worst case:

```text
(n1 - k + 1) · (n2 - k + 1)
```

Hash-based solution, worst case:

```text
(n1 - k + 1) + (n2 - k + 1)
```

The product becomes a sum.

For example, for $n_1 = 4,411,532, n_2=4,809,037$, the naive solution requires about $2\cdot 10^{13}$ comparisons, while the hash-based solution requires about $10^7$ comparisons.

## 3.3 Improving Memory Requirements

In [ ]:
# Code snippet: choose the shorter string for hashing

if len(s1) < len(s2):
    common_substring_hash(s1, s2, k)
else:
    common_substring_hash(s2, s1, k)

In [ ]:
# Code snippet: memory-saving hash solution

def common_substring_hash2(s1, s2, k):
    ''' find a common length k substring of s1 and s2
        using python built-in sets '''
    if len(s2) < len(s1):
        s1, s2 = s2, s1 # simultaneous assignment

    table = set()
    for i in range(len(s1)-k+1):
        table.add(s1[i:i+k])

    for i in range(len(s2)-k+1):
        if s2[i:i+k] in table:
            return s2[i:i+k]

    return None

In [ ]:
# Improve performance by hashing the shorter string, which creates a smaller hash table.

common_substring_hash2(Mycobacterium_tuberculosis,
Mycobacterium_leprae, 150)

## 3.3.2. Fingerprints

Instead of storing the whole k-mer, store a compact representation: its hash value.

This saves memory, but introduces a risk: two different strings may have the same fingerprint.

Therefore, a fingerprint match is only a **possible** match and must be verified.

In [ ]:
# Code snippet: common substring using fingerprints

def common_substring_fingerprint(s1, s2, k):
    ''' find a common length k substring of s1 and s2
        using python built-in sets and fingerprints to save memory '''
    if len(s2) < len(s1):
        s1, s2 = s2, s1

    table = set()

    for i in range(len(s1)-k+1):
        fingerprint = hash(s1[i:i+k])
        table.add(fingerprint)      # note that we are adding the fingerprint, not the substring itself
        # this saves memory, but can lead to false positives (collisions)

    for i in range(len(s2)-k+1):
        fingerprint = hash(s2[i:i+k])
        if fingerprint in table: # possible match
            if s2[i:i+k] in s1: # sanity check
                return s2[i:i+k]
            else:
                print("ALMOST A FALSE POSITIVE:", s2[i:i+k])
    return None

In [ ]:
# Code snippet: test fingerprint version
# This can still take time on real full genomes.

common_substring_fingerprint(Mycobacterium_tuberculosis, Salmonella_enterica, 1000)

## 3.4. Dictionaries: counting k-mers

A `set` can tell us whether a k-mer was seen before.

A `dict` can store a counter for each k-mer.

In [ ]:
# Code snippet: example dictionary for k-mers

st = "CCCTTGCTT"
k = 3
# Expected dictionary idea:
# {'GCT': 1, 'TTG': 1, 'CTT': 2, 'TGC': 1, 'CCT': 1, 'CCC': 1}

In [ ]:
# Code snippet: most frequent k-mer

def frequent_kmer(st, k):
    counters = dict() # key: k-mer, value: count
    most_freq = ""
    max_freq = 0

    for i in range(len(st)-k+1):
        if st[i:i+k] not in counters:
            counters[st[i:i+k]] = 1 # first time
        else:
            counters[st[i:i+k]] += 1 # found one more
            if max_freq < counters[st[i:i+k]]:
                max_freq = counters[st[i:i+k]]
                most_freq = st[i:i+k]

    print("k-mers and their counts:", counters)
    return most_freq, max_freq

In [ ]:
# Code snippet: small test for frequent_kmer
freq_k_mer = frequent_kmer("CCCTTGCTT", 3)
print("Most frequent k-mer:", freq_k_mer)

In [ ]:
# Code snippet: frequent k-mer in a genome

frequent_kmer(Mycobacterium_leprae, 3)

In [ ]:
# Code snippet: sanity check using Python's built-in count method

Mycobacterium_leprae.count('CGG')

## Challenge Yourself: Python `in` operator

The chapter also considers a shorter solution using Python’s `in` operator.

The inner loop is hidden inside Python’s implementation.

In [ ]:
# Code snippet: common substring using Python's in operator

def common_substring_better(s1, s2, k):
    ''' find a common substring of s1 and s2 of length k '''
    for i in range(len(s1)-k+1):
        if s1[i:i+k] in s2:
            return s1[i:i+k]
    return None

## Challenge Yourself: longest common substring

Toy example from the chapter:

```python
s1 = "GATTAGCCGTAGATTGA"
s2 = "AGGAAGGATGCCGTGAAA"
```

In [ ]:
# Code snippet: toy example strings

s1 = "GATTAGCCGTAGATTGA"
s2 = "AGGAAGGATGCCGTGAAA"

In [ ]:
# Code snippet: first stage of binary-search idea
# Note: the book text shows common_substring_hash4 here.
# The full version below uses common_substring_hash2.

def longest_common_substring_bsearch_stage1(s1, s2):
    ''' using binary search on the length '''
    print("Starting binary search on length of common substring...")
    k = 1
    while common_substring_hash2(s1, s2, k) != None:
        print(k, "found")
        k *= 2

    print(k, "not found")

In [ ]:
# Code snippet: longest common substring by doubling and binary search

def longest_common_substring_bsearch(s1, s2):
    ''' using binary search on the length '''
    print("Starting binary search on length of common substring...")
    k = 1
    while common_substring_hash2(s1, s2, k) != None:
        print(k, "found")
        k *= 2

    print(k, "not found")

    print("\nSearching for lengths between", k//2+1, "and", k-1, "...")
    longest = k//2
    low = k//2+1
    high = k-1
    while low <= high:
        mid = (low+high)//2
        res = common_substring_hash2(s1, s2, mid)
        if res == None:
            print(mid, "not found")
            high = mid-1
        else:
            print(mid, "found")
            low = mid+1
            longest = mid

    longest_ss = common_substring_hash2(s1, s2, longest)
    print("Longest common substring of length", longest, ":", longest_ss)

In [ ]:
# Code snippet: run on the toy example

longest_common_substring_bsearch(s1, s2)

## Challenge placeholder

Write a function:

```python
common_substring_3(st1, st2, st3, k)
```

It should return a common substring of length `k` that appears in all three strings.

## Summary

- Naïve search is simple but can be too slow.
- Python `set` supports fast membership testing because it is implemented using hash tables.
- Hash tables reduce lookup to a small set of candidates.
- Fingerprints reduce memory usage but require verification.
- Python `dict` is useful when we need to count occurrences.